**Loading Llama3.2 model**

In [21]:
!pip install -U langchain-community mypy_extensions
!pip install -U ddgs
!pip install langchain langchain-groq langchain-classic
!pip install python-dotenv
!pip install -U sentence-transformers faiss-cpu
!pip install cohere
!pip install -U rank_bm25

In [3]:
import os
import numpy as np
import pandas as pd
from langchain_groq import ChatGroq
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
llm=ChatGroq( model="llama-3.1-8b-instant",
    temperature=0,
    max_tokens=1024,
)

Generating response for the prompt

In [4]:
def build_llama3_prompt(messages):
    prompt = "<|begin_of_text|>"
    for m in messages:
        prompt+= f"<|start_header_id|>{m['role']}<|end_header_id|>\n\n{m['content']}<|eot_id|>"
    prompt+= "<|start_header_id|>assistant<|end_header_id|>\n\n"
    return prompt


def generate(messages, max_new_tokens=200, do_sample=True, temperature=1, top_p=0.25):
    prompt = build_llama3_prompt(messages)
    return llm.invoke(
        prompt,
        max_tokens=max_new_tokens,
        temperature=temperature if do_sample else 0.0,
        top_p=top_p,
    )

In [ ]:
messages = [
    {"role": "user", "content": "Who are the largest car manufacturers in 2023? Do they each makeEVs or not?"}
]
print(generate(messages))

In [ ]:
messages = [{"role": "user", "content": "Classify the text into neutral, negative or positive.\nText: I think the food was okay.\nSentiment:"}]
print(generate(messages))

In [ ]:
persona = "You are an expert in AI programming assistant.Help solving, writing, explaining any code to make the user's work easy.\n"
instruction= "Give a step by step answer for the user's question.If it's a coding, answer should be working code.\n"
context = "The assistance is used by the some developers.\n"
data_format = """1. Question summery.
2. Explaination
3. Code (if applicabel)
4. Conclusion\n"""
audience = "The user is beginner to intermediate programer.\n"
tone = "Friendly, professional, and concise.\n"
data = "Explain me a C program for sum of two numbers in a simple way with code."
full_prompt = persona + instruction + context + data_format + audience + tone + data
messages = [{"role": "user", "content": full_prompt}]
print(generate(messages,max_new_tokens=350))

**Chain-of-Thought — zero-shot version**

In [ ]:
zeroshot_cot_prompt = [
    {"role": "user", "content": "The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have? Let's think step-by-step."}
]
print(generate(zeroshot_cot_prompt, max_new_tokens=200))

**Tree-of-Thought**

In [ ]:
zeroshot_tot_prompt = [
    {"role": "user", "content": (
        "Imagine three different experts are answering this question. "
        "All experts will write down 1 step of their thinking, then share it with the group. "
        "Then all experts will go on to the next step, etc. "
        "If any expert realizes they're wrong at any point then they leave. "
        "The question is 'The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, "
        "how many apples do they have?' Make sure to discuss the results in short."
    )}
]
print(generate(zeroshot_tot_prompt, max_new_tokens=500))

**Output validation**

In [ ]:
one_shot_template = """Create a short character profile for an RPG game. Make sure to only use this format:
{
  "description": "A SHORT DESCRIPTION",
  "name": "THE CHARACTER'S NAME",
  "armor": "ONE PIECE OF ARMOR",
  "weapon": "ONE OR MORE WEAPONS"
}
"""
one_shot_prompt = [{"role": "user", "content": one_shot_template}]
output = generate(one_shot_prompt, max_new_tokens=150)
print(output)

**Chain:**

*Multiple chaining*

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain

template = """<s><|user|>
Create a title for a story about {summary}. Only return the title.<|end|>
<|assistant|>"""
title_prompt = PromptTemplate(template=template, input_variables=["summary"])
title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")

template = """<s><|user|>
Describe the main character of a story about {summary} with the title {title}.
Use only two sentences.<|end|>
<|assistant|>"""
character_prompt = PromptTemplate(
 template=template, input_variables=["summary","title"]
)
character = LLMChain(llm=llm, prompt=character_prompt, output_key="character")

template = """<s><|user|>
Create a story about {summary} with the title {title}. The main character is:
{character}. Only return the story and it cannot be longer than one paragraph.
<|end|>
<|assistant|>"""
story_prompt = PromptTemplate(
 template=template, input_variables=["summary", "title", "character"]
)
story = LLMChain(llm=llm, prompt=story_prompt, output_key="story")
llm_chain = title | character | story
llm_chain.invoke("a girl that lost her mother")


**Memory**: *Windowed Conversation Buffer*

In [ ]:
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_classic.chains import LLMChain
template = """<s><|user|>Current conversation:{chat_history}
{input_prompt}<|end|>
<|assistant|>"""
prompt = PromptTemplate(
 template=template,
 input_variables=["input_prompt", "chat_history"]
)

memory = ConversationBufferWindowMemory(k=2, memory_key="chat_history")
llm_chain = LLMChain(
 prompt=prompt,
 llm=llm,
 memory=memory
)
llm_chain.predict(input_prompt="Hi! My name is Manoj and I am 21 years old.What is 133+ 1?")
print("\n")
llm_chain.predict(input_prompt="I forgot my name. Do you know me")

2*Conversation Summary*

In [ ]:
from langchain_classic.memory import ConversationSummaryMemory

summary_prompt_template = """<s><|user|>Summarize the conversations and update
with the new lines.
Current summary:
{summary}
new lines of conversation:
{new_lines}
New summary:<|end|>
<|assistant|>"""
summary_prompt = PromptTemplate(
 input_variables=["new_lines", "summary"],
 template=summary_prompt_template
)
memory = ConversationSummaryMemory(
 llm=llm,
 memory_key="chat_history",
 prompt=summary_prompt
)
llm_chain = LLMChain(
 prompt=prompt,
 llm=llm,
 memory=memory
)
llm_chain.invoke({"input_prompt": "Hi! My name is Manoj S. What is 1 + 1?"})
llm_chain.invoke({"input_prompt": "What was the first question I asked?"})

**Agent tools**:

In [ ]:
from langchain_classic.agents import load_tools, Tool,AgentExecutor,create_react_agent
from langchain_community.tools import DuckDuckGoSearchResults

react_template = """Answer the following questions as best you can. You have
access to the following tools:
{tools}
Use the following format:
Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question
Begin!
Question: {input}
Thought:{agent_scratchpad}"""
prompt = PromptTemplate(
 template=react_template,
 input_variables=["tools", "tool_names","input", "agent_scratchpad"]
)
search = DuckDuckGoSearchResults()
search_tool =Tool(
 name="duckduck",
 description="A web search engine. Use this to as a search engine for general queries.",
 func=search.run,
)
tools=load_tools(["llm-math"],llm=llm)
tools.append(search_tool)
agent = create_react_agent(llm, tools, prompt)
agent_executor= AgentExecutor(agent=agent,tools=tools ,verbose=True, handle_parsing_errors=True,max_iterations=6,max_execution_time=90)
agent_executor.invoke({"input": "What is the current price of a MacBook Pro in USD? How much would it cost in EUR if the exchange rate is 0.85 EUR for 1 USD."})

**Sematic search**

In [8]:
text = """
Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind.
Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007.
Caltech theoretical physicist and 2017 Nobel laureate in Physics Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm.
Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles.
Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects.
Interstellar premiered on October 26, 2014, in Los Angeles.
In the United States, it was first released on film stock, expanding to venues using digital projectors.
The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014.
It received acclaim for its performances, direction, screenplay, musical score, visual effects, ambition, themes, and emotional weight.
It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics. Since its premiere, Interstellar gained a cult following, and now is regarded by many sci-fi experts as one of the best science-fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy Awards, winning Best Visual Effects, and received numerous other accolades"""

texts = text.split('.')
texts = [t.strip(' \n') for t in texts if t.strip(' \n')]

In [9]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeds = embedder.encode(texts, convert_to_numpy=True)
print(embeds.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(15, 384)


In [10]:
import faiss

dim = embeds.shape[1]
index = faiss.IndexFlatL2(dim)
print(index.is_trained)
index.add(np.float32(embeds))

True


In [11]:
def search(query, number_of_results=3):
    query_embed = embedder.encode([query], convert_to_numpy=True)
    distances, similar_item_ids = index.search(np.float32(query_embed), number_of_results)
    texts_np = np.array(texts)
    results = pd.DataFrame(data={'texts': texts_np[similar_item_ids[0]], 'distance': distances[0]})
    print(f"Query: '{query}'\nNearest neighbors:")
    return results

In [12]:
query = "how precise was the science"
results = search(query)
results

Query: 'how precise was the science'
Nearest neighbors:


,texts,distance
0,It has also received praise from many astronom...,1.048788
1,Caltech theoretical physicist and 2017 Nobel l...,1.336793
2,"Since its premiere, Interstellar gained a cult...",1.565360


*Reranking*

In [19]:
import cohere
co = cohere.Client(userdata.get('Cohere'))

In [20]:
query = "how precise was the science"
results = co.rerank(query=query, documents=texts, top_n=3, return_documents=True)
results.results

[RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics'), index=12, relevance_score=0.15232232),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014'), index=10, relevance_score=0.050354082),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan'), index=0, relevance_score=0.0350424)]

In [22]:
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction import _stop_words
import string

def bm25_tokenizer(text):
    tokenized_doc = []
    for token in text.lower().split():
        token = token.strip(string.punctuation)
        if len(token) > 0 and token not in _stop_words.ENGLISH_STOP_WORDS:
            tokenized_doc.append(token)
    return tokenized_doc

tokenized_corpus = [bm25_tokenizer(passage) for passage in texts]
bm25 = BM25Okapi(tokenized_corpus)

In [23]:
def keyword_and_reranking_search(query, top_k=3, num_candidates=10):
    print("Input question:", query)

    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)

    print("\nTop-3 lexical search (BM25) hits")
    for hit in bm25_hits[:top_k]:
        print(f"\t{hit['score']:.3f}\t{texts[hit['corpus_id']]}")

    docs = [texts[hit['corpus_id']] for hit in bm25_hits]

    print(f"\nTop-3 hits by rank-API ({len(docs)} BM25 hits re-ranked)")
    results = co.rerank(query=query, documents=docs, top_n=top_k, return_documents=True)
    for hit in results.results:
        print(f"\t{hit.relevance_score:.3f}\t{hit.document.text}")

In [24]:
keyword_and_reranking_search(query="how precise was the science")

Input question: how precise was the science

Top-3 lexical search (BM25) hits
	1.789	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	1.373	Caltech theoretical physicist and 2017 Nobel laureate in Physics Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar
	0.000	Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles

Top-3 hits by rank-API (10 BM25 hits re-ranked)
	0.152	It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics
	0.050	The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014
	0.035	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
